# ALL MAMBA -- Run All 7 Models (Eye Image -> HB Prediction)

Eye fundus/conjunctival image -> Haemoglobin (HB) estimation.
All model code is loaded from **this repo only** -- no external cloning.

| # | Model | Folder | Architecture |
|---|-------|--------|--------------|
| 1 | Mamba1 | 01_Mamba_Official | Selective SSM (Gu & Dao 2023) |
| 2 | Mamba2 | 01_Mamba_Official | Structured State Space (ICML 2024) |
| 3 | Mamba3 | 06_Mamba3_Minimal | MIMO + Rotary SSM (ICLR 2026) |
| 4 | VMamba | 02_VMamba | 2D Cross-Scan SSM (ICLR 2025) |
| 5 | MambaVision | 03_MambaVision | Hybrid Mamba+Transformer (CVPR 2025) |
| 6 | MedMamba | 04_MedMamba | Medical Vision Mamba (2024) |
| 7 | VSSD | 05_VSSD_Mamba2Vision | Vision Mamba-2 (ICCV 2025) |
| 8 | DSA-Mamba | 07_DSA_Mamba_Custom | Series Decomp + Cross-Attention |

> **Edit ONLY Section 1. Run all cells top to bottom.**


## Section 1 -- CONFIGURATION  (Edit only here)

In [ ]:
# ==============================================================
#              ALL VARIABLES -- EDIT HERE ONLY
# ==============================================================

# TASK:
#   "classification" -> Anemic (HB < threshold) vs Normal  [CrossEntropy]
#   "regression"     -> predict exact HB value in g/dL      [MSE]
#   "both"           -> classification AND regression        [default]
TASK = "both"

# DATASET PATHS (Kaggle)
IMAGE_DIR = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/left_eye"
CSV_PATH  = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/merge_excel_1.xlsx"
IMAGE_COL = "Patient ID"   # column whose value IS the image filename (no extension)
HB_COL    = "HB"           # column with hemoglobin measurement (g/dL)
HB_THRESH = 12.0           # below this -> Anemic (label 0), above -> Normal (label 1)

# TRAINING
EPOCHS      = 2             # epochs per model  (increase to 30-50 for real training)
BATCH_SIZE  = 4
LR          = 1e-4
VAL_SPLIT   = 0.2
NUM_WORKERS = 2
SEED        = 42

# Loss weights (only when TASK = "both")
CLS_WEIGHT  = 1.0           # weight for CrossEntropy loss
REG_WEIGHT  = 0.5           # weight for MSE regression loss

# Image size (all models use the same)
IMG_SIZE    = 224

# Which models to run (set False to skip)
RUN = {
    "Mamba1"      : True,
    "Mamba2"      : True,
    "Mamba3"      : True,
    "VMamba"      : True,
    "MambaVision" : True,
    "MedMamba"    : True,
    "VSSD"        : True,
    "DSA-Mamba"   : True,
}

# PATHS (auto-detected -- no need to change)
import os, sys
REPO_ROOT  = os.getcwd()                            # ALLMAMBA root
MODELS_DIR = os.path.join(REPO_ROOT, "MAMBA_MODELS")
print(f"TASK        : {TASK}")
print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"MODELS_DIR  : {MODELS_DIR}")
print(f"Models ON   : {[k for k,v in RUN.items() if v]}")


## Step 0 -- Clone Repo (run once on Kaggle, skip if already cloned)

In [ ]:
import subprocess

REPO_URL  = "https://github.com/junaidmaqbool/ALLMAMBA.git"
CLONE_DIR = "/kaggle/working/ALLMAMBA"

if not os.path.isdir(CLONE_DIR):
    print("Cloning ALLMAMBA ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)
    os.chdir(CLONE_DIR)
    REPO_ROOT  = CLONE_DIR
    MODELS_DIR = os.path.join(REPO_ROOT, "MAMBA_MODELS")
    print(f"Cloned. REPO_ROOT = {REPO_ROOT}")
else:
    print("Repo already cloned. Good to go.")


## Section 2 -- Install Dependencies

In [ ]:
import subprocess, sys

PKGS = [
    "mamba-ssm",      # compiled CUDA kernels for Mamba1/2 (Kaggle T4/P100)
    "causal-conv1d",  # required by mamba-ssm
    "einops",
    "timm",
    "pandas",
    "openpyxl",
    "scikit-learn",
    "matplotlib",
    "tqdm",
]
for pkg in PKGS:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(f"  {'OK  ' if r.returncode==0 else 'WARN'} {pkg}")
print("Dependencies done.")


## Section 3 -- Imports & Dataset

In [ ]:
import os, sys, math, time, warnings, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                              roc_auc_score, mean_absolute_error, f1_score)
from tqdm import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}   PyTorch: {torch.__version__}")
print(f"Task   : {TASK}")

# Make MODELS_DIR importable (common/pure_ssm.py lives there)
if MODELS_DIR not in sys.path:
    sys.path.insert(0, MODELS_DIR)


In [ ]:
class EyeHBDataset(Dataset):
    """Eye image dataset that returns (image_tensor, binary_label, hb_value)."""

    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        pid   = str(row[IMAGE_COL])
        hb    = float(row[HB_COL])
        label = 0 if hb < HB_THRESH else 1   # 0=Anemic, 1=Normal

        img_path = None
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ""]:
            p = os.path.join(IMAGE_DIR, pid + ext)
            if os.path.exists(p): img_path = p; break
        if img_path is None:
            raise FileNotFoundError(f"No image for Patient ID '{pid}' in {IMAGE_DIR}")

        img = Image.open(img_path).convert("RGB")
        if self.transform: img = self.transform(img)
        return (img,
                torch.tensor(label, dtype=torch.long),
                torch.tensor([[hb]],  dtype=torch.float32))


T_TRAIN = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
T_VAL = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])


def load_data():
    df = (pd.read_excel(CSV_PATH)
          if CSV_PATH.endswith((".xlsx", ".xls"))
          else pd.read_csv(CSV_PATH))
    lb = (df[HB_COL] < HB_THRESH).astype(int)
    tr_df, vl_df = train_test_split(df, test_size=VAL_SPLIT,
                                    random_state=SEED, stratify=lb)
    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=(DEVICE == "cuda"), drop_last=False)
    tl = DataLoader(EyeHBDataset(tr_df, T_TRAIN), shuffle=True,  **kw)
    vl = DataLoader(EyeHBDataset(vl_df, T_VAL),   shuffle=False, **kw)
    print(f"Total:{len(df)}  Train:{len(tr_df)}  Val:{len(vl_df)}")
    print(f"Anemic:{lb.sum()}/{len(df)} ({lb.mean()*100:.1f}%)")
    print(f"HB  mean={df[HB_COL].mean():.2f}  min={df[HB_COL].min():.2f}  max={df[HB_COL].max():.2f}")
    return tl, vl

TRAIN_LOADER, VAL_LOADER = load_data()


## Section 4 -- Training Utilities

In [ ]:
CE_FN  = nn.CrossEntropyLoss()
MSE_FN = nn.MSELoss()

def compute_loss(logits, hb_pred, labels, hb_true):
    """Returns (total_loss, cls_loss_val, reg_loss_val) based on TASK."""
    if   TASK == "classification":
        cl = CE_FN(logits, labels);  return cl, cl.item(), 0.0
    elif TASK == "regression":
        rl = MSE_FN(hb_pred, hb_true); return rl, 0.0, rl.item()
    else:   # "both"
        cl = CE_FN(logits, labels)
        rl = MSE_FN(hb_pred, hb_true)
        return CLS_WEIGHT * cl + REG_WEIGHT * rl, cl.item(), rl.item()


def run_epoch(model, loader, optimizer=None):
    """Run one train or validation epoch. Returns (metrics_dict, labels, preds, hb_true, hb_pred)."""
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_scores = [], [], []
    all_hbp,   all_hbt               = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels, hb_true in tqdm(loader, leave=False,
                                           desc="train" if training else "val  "):
            imgs, labels, hb_true = (imgs.to(DEVICE),
                                     labels.to(DEVICE),
                                     hb_true.to(DEVICE))
            logits, hb_pred = model(imgs)
            loss, _, _      = compute_loss(logits, hb_pred, labels, hb_true)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            probs = torch.softmax(logits, dim=1)
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_scores.extend(probs[:, 1].cpu().tolist())
            all_hbp.extend(hb_pred.detach().cpu().reshape(-1).tolist())
            all_hbt.extend(hb_true.cpu().reshape(-1).tolist())

    metrics = {"loss": total_loss / max(len(loader), 1)}
    if TASK in ("classification", "both"):
        metrics["acc"] = accuracy_score(all_labels, all_preds)
        metrics["f1"]  = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        try:    metrics["auc"] = roc_auc_score(all_labels, all_scores)
        except: metrics["auc"] = float("nan")
    if TASK in ("regression", "both"):
        at, ap = np.array(all_hbt), np.array(all_hbp)
        metrics["mae"]  = mean_absolute_error(at, ap)
        metrics["rmse"] = float(np.sqrt(np.mean((at - ap) ** 2)))
    return metrics, all_labels, all_preds, all_hbt, all_hbp


RESULTS = []   # filled by run_model()

def run_model(name: str, model: nn.Module):
    """Full training + evaluation loop for one model."""
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    sep    = "=" * 65
    print(f"\n{sep}\n  {name}   ({params/1e6:.2f}M params)   Task={TASK}\n{sep}")

    model  = model.to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=LR)
    sch    = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    hist   = {k: [] for k in ["tr_loss", "vl_loss", "acc", "auc", "f1", "mae", "rmse"]}
    t0     = time.time()

    vl_y = vl_p = None
    for ep in range(1, EPOCHS + 1):
        tr, *_                        = run_epoch(model, TRAIN_LOADER, opt)
        vl, vl_y, vl_p, vl_hbt, vl_hbp = run_epoch(model, VAL_LOADER)
        sch.step()

        hist["tr_loss"].append(tr["loss"])
        hist["vl_loss"].append(vl["loss"])
        for k in ["acc", "auc", "f1", "mae", "rmse"]:
            hist[k].append(vl.get(k, float("nan")))

        line = f"  Ep {ep}/{EPOCHS}  TL:{tr['loss']:.4f}  VL:{vl['loss']:.4f}"
        if TASK in ("classification", "both"):
            line += f"  Acc:{vl.get('acc',0):.3f}  AUC:{vl.get('auc',0):.3f}  F1:{vl.get('f1',0):.3f}"
        if TASK in ("regression", "both"):
            line += f"  MAE:{vl.get('mae',0):.2f}g/dL  RMSE:{vl.get('rmse',0):.2f}"
        print(line)

    elapsed = time.time() - t0

    if TASK in ("classification", "both") and vl_y is not None:
        print("\n  Classification Report (final epoch val):")
        print(classification_report(vl_y, vl_p, target_names=["Anemic","Normal"], zero_division=0))

    last = {k: (v[-1] if v else float("nan")) for k, v in hist.items()}
    RESULTS.append(dict(name=name, status="PASS", params=params, time_s=elapsed,
                        **last, error=""))

    # Plot curves
    ep_x  = range(1, EPOCHS + 1)
    ncols = 1 + (TASK in ("classification","both")) + (TASK in ("regression","both"))
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 3))
    if ncols == 1: axes = [axes]
    axes[0].plot(ep_x, hist["tr_loss"], label="Train")
    axes[0].plot(ep_x, hist["vl_loss"], label="Val")
    axes[0].set_title(f"{name} -- Loss"); axes[0].legend()
    col = 1
    if TASK in ("classification", "both"):
        axes[col].plot(ep_x, hist["acc"], label="Acc",  color="green")
        axes[col].plot(ep_x, hist["auc"], label="AUC",  color="purple", linestyle="--")
        axes[col].plot(ep_x, hist["f1"],  label="F1",   color="teal",   linestyle=":")
        axes[col].set_title("Classification"); axes[col].legend(); axes[col].set_ylim(0, 1)
        col += 1
    if TASK in ("regression", "both"):
        axes[col].plot(ep_x, hist["mae"],  label="MAE",  color="orange")
        axes[col].plot(ep_x, hist["rmse"], label="RMSE", color="red", linestyle="--")
        axes[col].set_title("HB Regression (g/dL)"); axes[col].legend()
    plt.suptitle(name, fontsize=11, y=1.02)
    plt.tight_layout()
    fname = f"result_{name.split()[0].replace('/','_')}.png"
    plt.savefig(fname, dpi=100, bbox_inches="tight"); plt.show()
    print(f"  Done in {elapsed:.0f}s  |  chart saved: {fname}")
    return model


def _fail(name: str, e: Exception):
    """Record a failed model run."""
    print(f"  {name} FAILED: {e}")
    traceback.print_exc()
    RESULTS.append(dict(name=name, status="FAIL", error=str(e),
                        params=0, time_s=0,
                        tr_loss=float("nan"), vl_loss=float("nan"),
                        acc=float("nan"), auc=float("nan"), f1=float("nan"),
                        mae=float("nan"), rmse=float("nan")))

print("Training utilities ready.")


## Section 5 -- sys.path helper

In [ ]:
def _add(folder: str):
    """Insert a MAMBA_MODELS sub-folder into sys.path (idempotent)."""
    p = os.path.join(MODELS_DIR, folder)
    if p not in sys.path:
        sys.path.insert(0, p)

print("sys.path helper ready.")


---
## Model 1 -- Mamba1 (Official SSM, Gu & Dao 2023)

In [ ]:
if RUN["Mamba1"]:
    _add("01_Mamba_Official")
    try:
        from eye_hb_model import build_mamba1
        mamba1_model = build_mamba1(IMG_SIZE, embed_dim=128, depth=4)
        run_model("Mamba1 (official SSM)", mamba1_model)
    except Exception as e:
        _fail("Mamba1", e)
else:
    print("Mamba1 skipped.")


---
## Model 2 -- Mamba2 / SSD (Dao & Gu, ICML 2024)

In [ ]:
if RUN["Mamba2"]:
    _add("01_Mamba_Official")
    try:
        from eye_hb_model import build_mamba2
        mamba2_model = build_mamba2(IMG_SIZE, embed_dim=128, depth=4)
        run_model("Mamba2 (SSD, ICML 2024)", mamba2_model)
    except Exception as e:
        _fail("Mamba2", e)
else:
    print("Mamba2 skipped.")


---
## Model 3 -- Mamba3 (MIMO + Rotary SSM, ICLR 2026)

In [ ]:
if RUN["Mamba3"]:
    # eye_hb_model for Mamba3 lives in 06_Mamba3_Minimal
    _add("06_Mamba3_Minimal")
    try:
        # Import specifically from the 06 folder
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_m3_eye", os.path.join(MODELS_DIR, "06_Mamba3_Minimal", "eye_hb_model.py"))
        m3_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(m3_eye)
        mamba3_model = m3_eye.build_model(IMG_SIZE, embed_dim=128, depth=4)
        run_model("Mamba3 (ICLR 2026 Oral)", mamba3_model)
    except Exception as e:
        _fail("Mamba3", e)
else:
    print("Mamba3 skipped.")


---
## Model 4 -- VMamba (2D Cross-Scan SSM, ICLR 2025)

In [ ]:
if RUN["VMamba"]:
    _add("02_VMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_vm_eye", os.path.join(MODELS_DIR, "02_VMamba", "eye_hb_model.py"))
        vm_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(vm_eye)
        vmamba_model = vm_eye.build_model(IMG_SIZE)
        run_model("VMamba (2D cross-scan, ICLR 2025)", vmamba_model)
    except Exception as e:
        _fail("VMamba", e)
else:
    print("VMamba skipped.")


---
## Model 5 -- MambaVision (NVIDIA, CVPR 2025)

In [ ]:
if RUN["MambaVision"]:
    _add("03_MambaVision")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_mv_eye", os.path.join(MODELS_DIR, "03_MambaVision", "eye_hb_model.py"))
        mv_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mv_eye)
        mambavision_model = mv_eye.build_model(IMG_SIZE)
        run_model("MambaVision (NVIDIA, CVPR 2025)", mambavision_model)
    except Exception as e:
        _fail("MambaVision", e)
else:
    print("MambaVision skipped.")


---
## Model 6 -- MedMamba (Medical Vision Mamba, 2024)

In [ ]:
if RUN["MedMamba"]:
    _add("04_MedMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_med_eye", os.path.join(MODELS_DIR, "04_MedMamba", "eye_hb_model.py"))
        med_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(med_eye)
        medmamba_model = med_eye.build_model(IMG_SIZE)
        run_model("MedMamba (medical imaging, 2024)", medmamba_model)
    except Exception as e:
        _fail("MedMamba", e)
else:
    print("MedMamba skipped.")


---
## Model 7 -- VSSD / VMAMBA2 (Mamba2 Vision, ICCV 2025)

In [ ]:
if RUN["VSSD"]:
    _add("05_VSSD_Mamba2Vision")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_vssd_eye", os.path.join(MODELS_DIR, "05_VSSD_Mamba2Vision", "eye_hb_model.py"))
        vssd_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(vssd_eye)
        vssd_model = vssd_eye.build_model(IMG_SIZE)
        run_model("VSSD (Mamba2 Vision, ICCV 2025)", vssd_model)
    except Exception as e:
        _fail("VSSD", e)
else:
    print("VSSD skipped.")


---
## Model 8 -- DSA-Mamba (First-Ronin Official)

In [ ]:
if RUN["DSA-Mamba"]:
    _add("07_DSA_Mamba_Custom")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_dsa_eye", os.path.join(MODELS_DIR, "07_DSA_Mamba_Custom", "eye_hb_model.py"))
        dsa_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(dsa_eye)
        dsamamba_model = dsa_eye.build_model(IMG_SIZE)
        run_model("DSA-Mamba (First-Ronin, official)", dsamamba_model)
    except Exception as e:
        _fail("DSA-Mamba", e)
else:
    print("DSA-Mamba skipped.")


---
## Section 6 -- Final Summary & Comparison Charts

In [ ]:
import math

def _f(v, fmt=".3f"):
    return format(v, fmt) if isinstance(v, float) and not math.isnan(v) else " n/a "

sep = "=" * 80
print(f"\n{sep}")
print(f"  FINAL RESULTS   Task={TASK}   Epochs={EPOCHS}   Batch={BATCH_SIZE}")
print(f"{sep}")
for r in RESULTS:
    line = f"  {r['name']:<44} {r['status']:<6}  {r.get('params',0)/1e6:.1f}M  {r.get('time_s',0):.0f}s"
    if TASK in ("classification","both"):
        line += f"  Acc:{_f(r.get('acc',float('nan')))}  AUC:{_f(r.get('auc',float('nan')))}  F1:{_f(r.get('f1',float('nan')))}"
    if TASK in ("regression","both"):
        line += f"  MAE:{_f(r.get('mae',float('nan')),'.2f')}  RMSE:{_f(r.get('rmse',float('nan')),'.2f')}"
    print(line)
print(f"{sep}")
passed = [r for r in RESULTS if r.get("status") == "PASS"]
failed = [r for r in RESULTS if r.get("status") != "PASS"]
print(f"  Passed: {len(passed)}   Failed: {len(failed)}")
if failed:
    print(f"  Failed models: {[r['name'] for r in failed]}")
    print("  Tip: pip install mamba-ssm causal-conv1d  (needs CUDA GPU on Kaggle T4)")
print(f"{sep}")


In [ ]:
ok = [r for r in RESULTS if r.get("status") == "PASS"]
if not ok:
    print("No models passed -- nothing to chart.")
else:
    names = [r["name"].split("(")[0].strip() for r in ok]

    if TASK in ("classification","both"):
        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        axes[0].barh(names, [r.get("acc", 0) for r in ok], color="steelblue")
        axes[0].set_title("Val Accuracy"); axes[0].set_xlim(0, 1)
        axes[1].barh(names, [r.get("auc", 0) for r in ok], color="purple")
        axes[1].set_title("Val AUC-ROC");  axes[1].set_xlim(0, 1)
        axes[2].barh(names, [r.get("f1",  0) for r in ok], color="teal")
        axes[2].set_title("Val F1-macro"); axes[2].set_xlim(0, 1)
        plt.tight_layout()
        plt.savefig("comparison_classification.png", dpi=100); plt.show()

    if TASK in ("regression","both"):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].barh(names, [r.get("mae",  0) for r in ok], color="orange")
        axes[0].set_title("Val MAE g/dL  (lower = better)")
        axes[1].barh(names, [r.get("rmse", 0) for r in ok], color="red")
        axes[1].set_title("Val RMSE g/dL (lower = better)")
        plt.tight_layout()
        plt.savefig("comparison_regression.png", dpi=100); plt.show()

    print("Comparison charts saved.")
